In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc

In [ ]:
df = pd.read_parquet('./Dataset/Dataset_small/train/train.parquet')

In [ ]:
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x='generated_by', y='word_count', data=df)
plt.xticks(rotation=45)
plt.title("Word Count per Model")
plt.show()

In [ ]:
df['generated_by'].value_counts()

In [ ]:
df['word_count'].describe()

# Remove Outliers

In [ ]:
def remove_outliers(df):
  word_counts = df['text'].str.split().str.len()
  Q1 = word_counts.quantile(0.25)
  Q3 = word_counts.quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df_removed_outliers = df[(word_counts >= lower_bound) & (word_counts <= upper_bound)].copy()
  return df_removed_outliers

In [ ]:
df_removed_outliers = remove_outliers(df)

In [ ]:
df_removed_outliers['text_length'] = df_removed_outliers['text'].apply(len)
df_removed_outliers['word_count'] = df_removed_outliers['text'].apply(lambda x: len(x.split()))

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x='generated_by', y='word_count', data=df_removed_outliers)
plt.xticks(rotation=45)
plt.title("Word Count per Model")
plt.show()

In [ ]:
df_removed_outliers['generated_by'].value_counts()

In [ ]:
df_removed_outliers['word_count'].describe()

In [ ]:
del df_removed_outliers

# Truncate

In [ ]:
def truncate(df, max_words = 500):
  df_truncated = df.copy()
  df_truncated['text'] = (df_truncated['text'].str.split().str[:max_words].str.join(" "))
  return df_truncated

In [ ]:
df_truncated = truncate(df)

In [ ]:
df_truncated['text_length'] = df_truncated['text'].apply(len)
df_truncated['word_count'] = df_truncated['text'].apply(lambda x: len(x.split()))

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x='generated_by', y='word_count', data=df_truncated)
plt.xticks(rotation=45)
plt.title("Word Count per Model")
plt.show()

In [ ]:
df_truncated['generated_by'].value_counts()

In [ ]:
df_truncated['word_count'].describe()

In [ ]:
del df_truncated

# Split

In [ ]:
def split_texts(df, max_length=500):
    df_split = df.copy()
    def chunk_words(text):
        words = text.split()
        return [" ".join(words[i:i + max_length]) for i in range(0, len(words), max_length)]
    df_split['text'] = df_split['text'].apply(chunk_words)
    df_exploded = df_split.explode('text', ignore_index=True)
    return df_exploded

In [ ]:
df_exploded = split_texts(df)

In [ ]:
df_exploded['text_length'] = df_exploded['text'].apply(len)
df_exploded['word_count'] = df_exploded['text'].apply(lambda x: len(x.split()))

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x='generated_by', y='word_count', data=df_exploded)
plt.xticks(rotation=45)
plt.title("Word Count per Model")
plt.show()

In [ ]:
df_exploded['generated_by'].value_counts()

In [ ]:
df_exploded['word_count'].describe()

In [ ]:
del df_exploded

# DownSampling

In [ ]:
def remove_duplicates(df):
    return df.drop_duplicates(subset=['text'], keep='first').copy()

In [ ]:
def balance(df):
    return pd.concat([grp.sample(df["generated_by"].value_counts().min(), random_state=42) for _, grp in df.groupby("generated_by")]).reset_index(drop=True)

# Data Cleaning

In [ ]:
def clean_noise(df):
    df_cleaned = df.copy()
    df_cleaned['text'] = df_cleaned['text'].str.replace(r'<[^<>]*>', '', regex=True)            # Remove HTML tags
    df_cleaned['text'] = df_cleaned['text'].str.replace(r'http\S+|www\.\S+', '', regex=True)    # Remove URLs
    df_cleaned['text'] = df_cleaned['text'].str.replace(r'\s+', ' ', regex=True).str.strip()    # Remove extra spaces caused by the deletions
    return df_cleaned

In [ ]:
def lowercase_text(df):
    df_lower = df.copy()
    df_lower['text'] = df_lower['text'].str.lower()
    return df_lower

In [ ]:
def remove_punctuation(df):
    df_nopunct = df.copy()
    # Keeps only letters, numbers, and spaces (\w is word characters, \s is whitespace)
    # The ^ means "NOT", so it replaces anything that is NOT a word/space with nothing
    df_nopunct['text'] = df_nopunct['text'].str.replace(r'[^\w\s]', '', regex=True)
    return df_nopunct

# Making real changes

In [ ]:
RAW_DIR   = "./Dataset"
CLEAN_DIR = "./Dataset_cleaned"

In [ ]:
SIZES     = ["small", "medium", "original"]
SPLITS    = ["train", "val", "test"]

In [ ]:
for size in SIZES:
    for split in SPLITS:
        print(f"\n--- Processing [{size.upper()} | {split.upper()}] ---")
        
        # Load raw data
        df = pd.read_parquet(f'{RAW_DIR}/Dataset_{size}/{split}/{split}.parquet')

        #df = clean_noise(df)
        #df = lowercase_text(df)
        #df = remove_punctuation(df)
        
        # 1. OUTLIERS PIPELINE
        out_dir = f"{CLEAN_DIR}/remove_outliers/Dataset_{size}/{split}"
        os.makedirs(out_dir, exist_ok=True)
        
        df_outliers = balance(remove_duplicates(remove_outliers(df)))
        df_outliers.to_parquet(f"{out_dir}/{split}.parquet", index=False)
        print(f"Outliers Removed -> shape={df_outliers.shape}, samples/class={df_outliers['generated_by'].value_counts().iloc[0]}")
        del df_outliers
        
        # 2. TRUNCATE PIPELINE
        out_dir = f"{CLEAN_DIR}/truncate/Dataset_{size}/{split}"
        os.makedirs(out_dir, exist_ok=True)
        
        df_trunc = balance(remove_duplicates(truncate(df)))
        df_trunc.to_parquet(f"{out_dir}/{split}.parquet", index=False)
        print(f"Truncated        -> shape={df_trunc.shape}, samples/class={df_trunc['generated_by'].value_counts().iloc[0]}")
        del df_trunc
        
        # 3. SPLIT PIPELINE
        out_dir = f"{CLEAN_DIR}/split/Dataset_{size}/{split}"
        os.makedirs(out_dir, exist_ok=True)
        
        df_exp = balance(remove_duplicates(split_texts(df)))
        df_exp.to_parquet(f"{out_dir}/{split}.parquet", index=False)
        print(f"Split/Exploded   -> shape={df_exp.shape}, samples/class={df_exp['generated_by'].value_counts().iloc[0]}")
        del df_exp
        
        del df 
        gc.collect()

# Experiments

In [ ]:
def strict_windowing(df, exact_length=150):
    df_strict = df.copy()
    
    # 1. Drop rows that are too short to meet the exact length
    word_counts = df_strict['text'].str.split().str.len()
    df_strict = df_strict[word_counts >= exact_length]
    
    # 2. Truncate everything else to EXACTLY exact_length
    df_strict['text'] = (
        df_strict['text']
        .str.split()
        .str[:exact_length]
        .str.join(" ")
    )
    
    # 3. Now run your normal row balancer
    # (Since every row is 150 words, balancing rows = balancing words)
    return balance(df_strict)

In [ ]:
import pandas as pd

def concatenate_short_texts(df, target_length=200):
    # 1. Get the unique classes and the total number of them
    unique_classes = df['generated_by'].unique()
    num_classes = len(unique_classes)
    
    # The helper function remains exactly the same
    def glue_texts(sub_df):
        glued_texts = []
        current_text = []
        current_length = 0
        
        for text in sub_df['text']:
            words = str(text).split()
            current_text.extend(words)
            current_length += len(words)
            
            if current_length >= target_length:
                glued_texts.append(" ".join(current_text[:target_length]))
                current_text = []
                current_length = 0
                
        return pd.DataFrame({'text': glued_texts})

    # 2. Loop through each class dynamically
    processed_dfs = []
    
    for cls in unique_classes:
        # Isolate the data for this specific class
        df_class = df[df['generated_by'] == cls].copy()
        
        # Glue the texts together
        df_class_glued = glue_texts(df_class)
        
        # Re-assign the correct label to the glued texts
        df_class_glued['generated_by'] = cls
        
        # Add this finished class to our master list
        processed_dfs.append(df_class_glued)

    # 3. Recombine all the separate dataframes back into one
    df_combined = pd.concat(processed_dfs).reset_index(drop=True)
    
    # Safety Check
    if df_combined.empty:
        raise ValueError(f"The combined dataframe is empty! Your target_length ({target_length}) might be too high.")
        
    # 4. Finally, balance the row counts across all the classes
    return balance(df_combined)

In [ ]:
def pad_n_truncate(df, max_words=500, pad_token="<PAD>"):
    df_truncated = df.copy()
    
    # Helper function to process each row
    def adjust_text(text):
        words = str(text).split()
        
        # If it's too long, truncate it
        if len(words) >= max_words:
            return " ".join(words[:max_words])
            
        # If it's too short, pad it
        else:
            padding_needed = max_words - len(words)
            padded_words = words + [pad_token] * padding_needed
            return " ".join(padded_words)
            
    # Apply the logic to the text column
    df_truncated['text'] = df_truncated['text'].apply(adjust_text)
    
    return df_truncated

In [ ]:
def balance_by_upsampling(df):
    # Find the size of the LARGEST class instead of the smallest
    max_class_size = df["generated_by"].value_counts().max()
    
    return pd.concat([
        # replace=True allows Pandas to pick the same row multiple times
        grp.sample(max_class_size, replace=True, random_state=42) 
        for _, grp in df.groupby("generated_by")
    ]).reset_index(drop=True)

In [ ]:
df = pd.read_parquet('./Dataset/Dataset_small/train/train.parquet')
x = df.shape[0]
x

In [ ]:
df1 = strict_windowing(df)
y = df1.shape[0]
y

In [ ]:
df2 = concatenate_short_texts(df)
z = df2.shape[0]
z

In [ ]:
df3 = pad_n_truncate(df)
w = df3.shape[0]
w

In [ ]:
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

In [ ]:
loss = (x - y)/x
loss

In [ ]:
loss = (x - z)/x
loss

In [ ]:
x = list(input().split())
y = list(input().split())

In [ ]:
for i in x:
    for j in y:
        print(i, j, sep = '_', end = ".py\n")

In [ ]:
import pandas as pd
import os
import gc
gc.collect()

RAW_DIR = "./Dataset"
CLEAN_DIR = "./Dataset_cleaned/remove_outliers_upsample"
SIZES = ["small", "medium", "original"]
SPLITS = ["train", "val", "test"]

def remove_duplicates(df):
    return df.drop_duplicates(subset=['text'], keep='first').copy()

def upsample(df):
    max_size = df["generated_by"].value_counts().max()
    return pd.concat([grp.sample(max_size, replace=True, random_state=42) for _, grp in df.groupby("generated_by")]).reset_index(drop=True)

def remove_outliers(df):
    word_counts = df['text'].str.split().str.len()
    Q1, Q3 = word_counts.quantile(0.25), word_counts.quantile(0.75)
    IQR = Q3 - Q1
    return df[(word_counts >= (Q1 - 1.5 * IQR)) & (word_counts <= (Q3 + 1.5 * IQR))].copy()

for size in SIZES:
    for split in SPLITS:
        df = pd.read_parquet(f'{RAW_DIR}/Dataset_{size}/{split}/{split}.parquet')
        out_dir = f"{CLEAN_DIR}/Dataset_{size}/{split}"
        os.makedirs(out_dir, exist_ok=True)
        
        df_final = upsample(remove_duplicates(remove_outliers(df)))
        df_final.to_parquet(f"{out_dir}/{split}.parquet", index=False)
        gc.collect()